In [1]:
import pandas as pd
import numpy as np
import math

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv('https://github.com/Jeevita-R/ML_datasets/raw/refs/heads/main/House_price_prediction.csv')
df

,Unnamed: 0,Suburb,Rooms,Type,Method,SellerG,Regionname,Propertycount,Distance,CouncilArea,Bedroom2,Bathroom,Car,Landsize,BuildingArea,Price
0,1,Abbotsford,2,h,S,Biggin,Northern Metropolitan,4019.0,2.5,Yarra City Council,2.0,1.0,1.0,202.000000,160.2564,1480000.0
1,2,Abbotsford,2,h,S,Biggin,Northern Metropolitan,4019.0,2.5,Yarra City Council,2.0,1.0,0.0,156.000000,79.0000,1035000.0
2,4,Abbotsford,3,h,SP,Biggin,Northern Metropolitan,4019.0,2.5,Yarra City Council,3.0,2.0,0.0,134.000000,150.0000,1465000.0
3,5,Abbotsford,3,h,PI,Biggin,Northern Metropolitan,4019.0,2.5,Yarra City Council,3.0,2.0,1.0,94.000000,160.2564,850000.0
4,6,Abbotsford,4,h,VB,Nelson,Northern Metropolitan,4019.0,2.5,Yarra City Council,3.0,1.0,2.0,120.000000,142.0000,1600000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27239,34852,Yarraville,4,h,PI,Jas,Western Metropolitan,6543.0,6.3,Maribyrnong City Council,4.0,1.0,3.0,593.000000,160.2564,1480000.0
27240,34853,Yarraville,2,h,SP,Sweeney,Western Metropolitan,6543.0,6.3,Maribyrnong City Council,2.0,2.0,1.0,98.000000,104.0000,888000.0
27241,34854,Yarraville,2,t,S,Jas,Western Metropolitan,6543.0,6.3,Maribyrnong City Council,2.0,1.0,2.0,220.000000,120.0000,705000.0
27242,34855,Yarraville,3,h,SP,hockingstuart,Western Metropolitan,6543.0,6.3,Maribyrnong City Council,0.0,0.0,0.0,593.598993,160.2564,1140000.0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27244 entries, 0 to 27243
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Unnamed: 0     27244 non-null  int64  
 1   Suburb         27244 non-null  object 
 2   Rooms          27244 non-null  int64  
 3   Type           27244 non-null  object 
 4   Method         27244 non-null  object 
 5   SellerG        27244 non-null  object 
 6   Regionname     27244 non-null  object 
 7   Propertycount  27244 non-null  float64
 8   Distance       27244 non-null  float64
 9   CouncilArea    27244 non-null  object 
 10  Bedroom2       27244 non-null  float64
 11  Bathroom       27244 non-null  float64
 12  Car            27244 non-null  float64
 13  Landsize       27244 non-null  float64
 14  BuildingArea   27244 non-null  float64
 15  Price          27244 non-null  float64
dtypes: float64(8), int64(2), object(6)
memory usage: 3.3+ MB


In [4]:
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

for col in df.columns:
    if df[col].nunique() < 10 and df[col].dtype != 'object':
        print(col, df[col].nunique(), "-> treated as categorical")
        categorical_cols.append(col)

print("Final Categorical Columns:", categorical_cols)

Final Categorical Columns: ['Suburb', 'Type', 'Method', 'SellerG', 'Regionname', 'CouncilArea']


In [5]:
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

df_encoded.head()

,Unnamed: 0,Rooms,Propertycount,Distance,Bedroom2,Bathroom,Car,Landsize,BuildingArea,Price,...,CouncilArea_Moorabool Shire Council,CouncilArea_Moreland City Council,CouncilArea_Nillumbik Shire Council,CouncilArea_Port Phillip City Council,CouncilArea_Stonnington City Council,CouncilArea_Whitehorse City Council,CouncilArea_Whittlesea City Council,CouncilArea_Wyndham City Council,CouncilArea_Yarra City Council,CouncilArea_Yarra Ranges Shire Council
0,1,2,4019.0,2.5,2.0,1.0,1.0,202.0,160.2564,1480000.0,...,False,False,False,False,False,False,False,False,True,False
1,2,2,4019.0,2.5,2.0,1.0,0.0,156.0,79.0000,1035000.0,...,False,False,False,False,False,False,False,False,True,False
2,4,3,4019.0,2.5,3.0,2.0,0.0,134.0,150.0000,1465000.0,...,False,False,False,False,False,False,False,False,True,False
3,5,3,4019.0,2.5,3.0,2.0,1.0,94.0,160.2564,850000.0,...,False,False,False,False,False,False,False,False,True,False
4,6,4,4019.0,2.5,3.0,1.0,2.0,120.0,142.0000,1600000.0,...,False,False,False,False,False,False,False,False,True,False


In [6]:
X = df_encoded.drop('Price', axis=1)
y = df_encoded['Price']

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import math

In [10]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = math.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    n = X_test.shape[0]
    p = X_test.shape[1]
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

    return mae, mse, rmse, r2, adj_r2

In [11]:
lin_model = LinearRegression()

lin_results = evaluate_model(
    lin_model,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)

print("Linear Regression Results:")
print("MAE:", lin_results[0])
print("MSE:", lin_results[1])
print("RMSE:", lin_results[2])
print("R2:", lin_results[3])
print("Adjusted R2:", lin_results[4])

Linear Regression Results:
MAE: 233221.93624072854
MSE: 130996983379.99518
RMSE: 361935.05409119354
R2: 0.682684184977251
Adjusted R2: 0.6324183371796861


In [12]:
ridge_model = Ridge(alpha=1.0)

ridge_results = evaluate_model(
    ridge_model,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)

print("\nRidge Regression Results:")
print("MAE:", ridge_results[0])
print("MSE:", ridge_results[1])
print("RMSE:", ridge_results[2])
print("R2:", ridge_results[3])
print("Adjusted R2:", ridge_results[4])


Ridge Regression Results:
MAE: 233199.7552370946
MSE: 130981989471.86215
RMSE: 361914.3399643929
R2: 0.6827205049294884
Adjusted R2: 0.6324604105583358


In [13]:
lasso_model = Lasso(alpha=0.1)

lasso_results = evaluate_model(
    lasso_model,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)

print("\nLasso Regression Results:")
print("MAE:", lasso_results[0])
print("MSE:", lasso_results[1])
print("RMSE:", lasso_results[2])
print("R2:", lasso_results[3])
print("Adjusted R2:", lasso_results[4])


Lasso Regression Results:
MAE: 233179.5643268766
MSE: 130940186872.09921
RMSE: 361856.5832924685
R2: 0.6828217639483727
Adjusted R2: 0.632577709970388


c:\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.406e+15, tolerance: 8.961e+11
  model = cd_fast.enet_coordinate_descent(
